# Žmogus grandinėje: veiksmo prieš veiksmo vartai, rizikos sluoksniavimas ir audito žurnalas

Šioje pamokoje pateiktame README pristatomas Žmogus grandinėje su trumpu fragmentu, kuris po to, kai agentas jau pateikė atsakymą, prašo vartotojo `PATVIRTINTI` arba `ATMESTI`. Šis modelis yra geras pradžios taškas, tačiau gamybos HITL įgyvendinimai dažnai reikalauja dar trijų papildomų dalių:

1. **Veiksmo prieš veiksmo vartai**, kurie veikia **prieš** agentui atliekant rizikingą žingsnį, kad būtų kontroliuojamos sąnaudos, negrįžtamumas ir delsimas.
2. **Rizikos sluoksniavimas**, kad mažos rizikos veiksmai būtų atliekami automatiškai, vidutinės rizikos veiksmai būtų patvirtinami paketais, o tik didelės rizikos veiksmai būtų blokuojami žmogaus.
3. **Audito žurnalas ir redagavimo ciklas**, kad kiekvienas vartų sprendimas būtų įrašomas JSONL formatu, o atmetimas iš naujo nukreiptų agentą su struktūrizuota priežastimi, o ne tik atspausdintų `Redaguojama...`.

Šis užrašas kuria kiekvieną iš šių dalykų, pasinaudodamas tais pačiais pagrindais kaip `06-system-message-framework.ipynb`. Jis veikia pilnu ciklu režimu `DEMO_MODE = True` (nereikia interaktyvaus įvesties) arba su realiais `input()` raginimais, kai `DEMO_MODE = False`. Pastaba: DEMO_MODE režime trečio tikslo pakartojimas yra suprogramuotas, todėl ciklo mechanika matoma visa apimtimi. Tikras redagavimu paremtas atkategorinimas reikalauja `DEMO_MODE = False` ir operatoriaus.

**Už šios tematikos ribų (sprendžiama kitose pamokose):** autentifikacija ir prieigos kontrolė (Pamokos 06 README grėsmė 2), įrankių iškvietimų tarpinis sluoksnis (Pamoka 14 MAF giluminis tyrimas), kelių agentų debatai.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Modelis 1: Veiksmo prieš vartus

README HITL fragmentas pirmiausia iškviečia agentą, po to prašo vartotojo patvirtinti rezultatą. Tai yra **po veiksmo** srautas. Agentas jau buvo įvykdytas, todėl LLM skambučio kaina jau sumokėta, ir bet koks šalutinis poveikis (nusiųstas el. laiškas, įrašyta duomenų bazės eilutė, paskelbtas komentaras) jau įvyko.

**Prieš veiksmo** srautas įterpia vartus prieš agentui atliekant rizikingą veiksmą. Agentas pasiūlo veiksmą, vartai nusprendžia, ar vykdyti, ir tik patvirtinus įvyksta šalutinis poveikis.

| Aspektas | Po veiksmo patvirtinimas (README fragmentas) | Prieš veiksmo vartai (šis užrašų knygutė) |
|---|---|---|
| Kada vyksta patvirtinimas? | Po to, kai agentas pagamino rezultatą | Prieš vykdant bet kokį šalutinį poveikį |
| LLM kaina atmetus | Jau sumokėta | Sumokėta tik už pasiūlymą, ne už veiksmą |
| Nepakeičiami šalutiniai poveikiai | Galimi (veiksmas jau įvyko) | Užkirstas kelias |
| Auditavimo aiškumas | Patvirtinimas yra išvesties sakinys | Patvirtinimas yra JSON įrašas su laiko žyma, veiksmu, priežastimi |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Šablonas 2: Rizikos sluoksniavimas

Ne kiekvienas veiksmas reikalauja žmogaus patvirtinimo. Tik skaitymui skirtas užklausimas viešajam API yra kitokios reikšmės nei el. pašto siuntimas klientui. Abi situacijas vertinti vienodai reiškia veltui eikvoti operatoriaus dėmesį ir sulėtinti agentą.

Paprastas 3 sluoksnių modelis:

| Sluoksnis | Pavyzdžiai | Patvirtinimo eiga |
|---|---|---|
| `žemas` (tik skaitymui) | Ieškoti žinių bazėje, ieškoti skrydžių variantų, gauti viešą tinklalapį | Automatinis vykdymas, registruojama audito tikslais |
| `vidutinis` (nesudėtingas pokytis) | Talpinti rezultatą talpykloje, didinti skaitiklį, suplanuoti priminimą | Automatinis vykdymas, bet apžvalga vykdoma kasdien grupuojant |
| `aukštas` (išorinės reikšmės arba neatšaukiama) | Siųsti el. laišką, nuskaičiuoti kortelę, skelbti viešame kanale | Užblokuojama laukiant žmogaus patvirtinimo |

Tai yra vienas sluoksniavimo būdas. Veikiančiose sistemose dažnai taikomi smulkesni sluoksniai (pvz., AWS IAM leidimų lygiai, prieigos pagrįstos vaidmenimis sluoksniai). Žemiau pateikta 3 sluoksnių versija yra mažiausia naudinga versija agentui, kuris kombinuoja tik skaitymui skirtus veiksmus ir veiksmus su šalutinėmis pasekmėmis.

Žemiau pateiktas klasifikatorius naudoja raktinių žodžių heuristiką, kad demonstracija būtų deterministinė ir pigi. Veikiančioje sistemoje tai pakeistumėte išmoktu klasifikatoriumi arba politikos varikliu.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Modelis 3: Audito žurnalas ir redagavimo ciklas

`print("Response approved.")` nėra audito žurnalas. Norint užtikrinti pasitikėjimą, kiekvienas sprendimas prie vartų turėtų būti įrašytas kaip struktūruotas įvykis, kuriuo vėliau galite atlikti užklausą, pakartotinai paleisti arba pridėti prie incidento peržiūros.

Du elementai:

1. **Tik rašymui skirtas JSONL.** Viena eilutė kiekvienam sprendimui, su laiko žyma, veiksmais, lygiu, sprendimu, priežastimi. Lengva ieškoti, lengva vėliau siųsti į tikrą žurnalų saugyklą.
2. **Redagavimo ciklas atmetimo atveju.** Kai vartai grąžina `deny`, agentas pats iš naujo užduoda klausimą su atmetimo priežastimi kontekste, kad kitas pasiūlymas galėtų išvengti problemos.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Papildomi ištekliai

Keletas kitų viešųjų projektų įgyvendina šių HITL modelių variantus. Palyginkite požiūrius, kad rastumėte, kas tinka jūsų aplinkai:

- **LangChain** žmogaus valdomų įrankių apvyniojimai ([dokumentacija](https://python.langchain.com/docs/integrations/tools/human_tools)): tiesioginiai įrankių apvyniojimai, kurie sustabdo vykdymą žmogaus įvesties.
- **AutoGen** `UserProxyAgent` ([v0.2 dokumentacija](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ pertvarkė šią struktūrą): naudoja agento vaidmenį specialiai žmogaus atvaizdavimui daugiaagentinėse pokalbiuose.
- **Semantic Kernel** funkcijų filtrai ([dokumentacija](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): tarpinės programos sluoksnis, kuris veikia prie kiekvieno funkcijos kvietimo, tinkamas srautų valdymo logikai.

Kiekvienas projektas skirtingai tvarko tris posistemius: LangChain apvynioja juos kaip įrankius, AutoGen naudoja agento vaidmenį, Semantic Kernel taiko tarpinės programos filtrus. Perskaitykite vieną ar du įgyvendinimus nuo pradžios iki pabaigos, prieš pasirinkdami dizainą savo agentui.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
